In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [ ]:
from src.config import SimConfig, EnvConfig
from src.utils.data_processing import load_and_cache_entire_fleet
from src.utils.evaluation import VoyageBenchmarker, print_markdown_table
from src.utils.plotting import plot_simulation_dashboard, plot_benchmarker_results, plot_markov_matrix 
from src.solvers import HybridSDPSolver, BaselineSDPSolver
from src.controllers import (
    build_approach,
    HybridFCLockedControl,
    HybridPolicyControl,
    HybridValueControl,
    BaselineConstantControl,
    BaselineThresholdControl,
    BaselineSDPControl
)

In [ ]:
env = EnvConfig()
config = SimConfig()

fleet_data = load_and_cache_entire_fleet(env)
exclude_days = [] 
benchmarker = VoyageBenchmarker(fleet_data, env, config, exclude_days)

In [ ]:
fc_only_approaches = {
    "MacroConstantControl": build_approach(
        controller_cls=BaselineConstantControl,
        is_macro=True
    ),
    "ConstantControl": build_approach(
        controller_cls=BaselineConstantControl
    ),
    "MacroThresholdControl": build_approach(
        controller_cls=BaselineThresholdControl,
        is_macro=True
    ),
    "ThresholdControl": build_approach(
        controller_cls=BaselineThresholdControl
    ),
    "MacroSDPControl": build_approach(
        controller_cls=BaselineSDPControl,
        solver_cls=BaselineSDPSolver,
        is_macro=True
    ),
    "SDPControl": build_approach(
        controller_cls=BaselineSDPControl,
        solver_cls=BaselineSDPSolver
    ),
}

hybrid_approaches = {
    "MacroFCLocked": build_approach(
        controller_cls=HybridFCLockedControl,
        solver_cls=HybridSDPSolver,
        is_macro=True
    ),
    "FCLocked": build_approach(
        controller_cls=HybridFCLockedControl,
        solver_cls=HybridSDPSolver,
    ),
    "MacroPolicy": build_approach(
        controller_cls=HybridPolicyControl,
        solver_cls=HybridSDPSolver,
        is_macro=True
    ),
    "Policy": build_approach(
        controller_cls=HybridPolicyControl,
        solver_cls=HybridSDPSolver,
    ),
    "MacroValue": build_approach(
        controller_cls=HybridValueControl,
        solver_cls=HybridSDPSolver,
        is_macro=True
    ),
    "Value": build_approach(
        controller_cls=HybridValueControl,
        solver_cls=HybridSDPSolver,
    ),
}

In [ ]:
train_days = [1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
test_day = 4
approaches = hybrid_approaches
#approaches = fc_only_approaches

mc_model, _, _ = benchmarker._get_or_compute_models(train_days, solver_cls=None, horizon_length=1)
plot_markov_matrix(mc_model)

In [ ]:
print("--- APPROACH A: HAND-PICKED EVALUATION ---")

report = benchmarker.compare_approaches(approaches, train_days, test_day)
print_markdown_table(report.summary)

for app in approaches:
    plot_simulation_dashboard(report.get_telemetry(app), benchmarker.config, title=f"Day {test_day} - {app}", indiv=False)

In [ ]:
print("\n--- APPROACH B: LEAVE-ONE-OUT (Discrete Tracking vs Baseline) ---")

for app in approaches:
    report = benchmarker.run_leave_one_out(approaches[app])
    plot_benchmarker_results(report.summary, title=f"Leave-One-Out Cross Validation ({app})", plot_type='bar')
    print_markdown_table(report.summary)

In [ ]:
print("\n--- APPROACH C: FORWARD CHAINING (Learning Curve) ---")

for app in approaches:
    report = benchmarker.run_forward_chaining(approaches[app])
    plot_benchmarker_results(report.summary, title=f"Forward Chaining Learning Curve ({app})", plot_type='line')
    print_markdown_table(report.summary)